# Day 2: LLM Preprocessing (Vietnamese) — v2

LLM chi tao **Mo ta + Thong so**. Title/category/brand lay tu data goc, ghep thanh summary.

**Pipeline:** Load HF Hub -> Assign IDs -> Test single item -> Batch 120K -> Check results -> Build prompts -> Clean up -> Push HF Hub

**Model:** `groq/openai/gpt-oss-20b` | **Chi phi:** ~$9-10 | **Input:** `SeanSunny/items_raw_tv_v4` | **Output:** `SeanSunny/items_tv_v4`

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
import os
from groq import Groq
from pricer_vi.batch import Batch
from pricer_vi.items import Item
from pricer_vi.preprocessor import SYSTEM_PROMPT, build_summary

load_dotenv(override=True)
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

## 1. Load dataset from HuggingFace Hub

In [2]:
dataset = "SeanSunny/items_raw_tv_v4"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 120,000 items
title='Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870' category='Điện Tử - Công Nghệ' price=2376000 full='Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC\nKính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH... | Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH VỚI TẤT CẢ MÃ MÁY CÓ TRONG TÊN SẢN PHẨM THÔNG SỐ KỸ THUẬT Dùng cho tất cả các mã máy có trong tên sản phẩm Công suất: Tiêu chuẩn pin theo máy chênh lệch +/- 5% Điện Áp: Tiêu chuẩn Số Cell: Tiêu chuẩn Loại Pin: Li-on. Thời gian sử dụng cho một lần sạc đầy: 2h – 4h – 6h tùy số Cell và đời máy Hàng mới full box 100%, hoàn toàn tương thích với

In [3]:
# Assign IDs (required for batch custom_id mapping)
for index, item in enumerate(items):
    item.id = index

print(f"Assigned IDs 0 to {len(items)-1}")

Assigned IDs 0 to 119999


In [4]:
# Inspect raw data
print(f"Title: {items[0].title}")
print(f"Category: {items[0].category}")
print(f"Price: {items[0].price:,} VND")
print(f"Brand: {items[0].brand}")
print(f"\nFull text ({len(items[0].full)} chars):")
print(items[0].full[:500])

Title: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Category: Điện Tử - Công Nghệ
Price: 2,376,000 VND
Brand: TEEMO PC

Full text (3082 chars):
Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC
Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH... | Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH VỚI TẤT CẢ MÃ


## 2. Test single item

SYSTEM_PROMPT chi yeu cau LLM tra ve 2 truong: Mo ta + Thong so.
Title/category/brand lay tu data goc qua `build_summary()`.

In [5]:
print("SYSTEM_PROMPT:")
print(SYSTEM_PROMPT)

SYSTEM_PROMPT:
Tạo mô tả ngắn gọn cho một sản phẩm. Chỉ trả lời đúng 2 dòng theo định dạng sau. Không bao gồm mã sản phẩm.
Mô tả: 1 câu mô tả sản phẩm
Thông số: 1 câu về tính năng nổi bật


In [6]:
# Test LLM on 1 item, then build full summary
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")
llm_text = response.choices[0].message.content

print("=== LLM response (only Mo ta + Thong so) ===")
print(llm_text)
print()
print("=== Final summary (with original title/category/brand) ===")
summary = build_summary(items[0], llm_text)
print(summary)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

=== LLM response (only Mo ta + Thong so) ===
Mô tả: Pin tương thích cho Laptop Dell Vostro 14 5459, chất liệu Li‑ion, công suất chuẩn, thời gian sạc nhanh từ 2‑6 giờ tùy cell.  
Thông số: Sạc ổn định, dung lượng 100 % trong 6‑12 tháng bảo hành, hỗ trợ tất cả các mô hình Dell Vostro.

=== Final summary (with original title/category/brand) ===
Tiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Danh mục: Điện Tử - Công Nghệ
Thương hiệu: TEEMO PC
Mô tả: Pin tương thích cho Laptop Dell Vostro 14 5459, chất liệu Li‑ion, công suất chuẩn, thời gian sạc nhanh từ 2‑6 giờ tùy cell.  
Thông số: Sạc ổn định, dung lượng 100 % trong 6‑12 tháng bảo hành, hỗ trợ tất cả các mô hình Dell Vostro.

Input tokens: 1119
Output tokens: 97
Cost: 0.011 cents


## 3. Test batch nho (10 items) — Optional

Tao JSONL, upload Groq, submit batch, fetch results. Bo qua neu da test o v1.

In [7]:
BATCH_MODEL = "openai/gpt-oss-20b"

def make_jsonl(item):
    body = {
        "model": BATCH_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
        "reasoning_effort": "low",
    }
    line = {
        "custom_id": str(item.id),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": body,
    }
    return json.dumps(line, ensure_ascii=False)

def make_file(start, end, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

# Tao test file 10 items
os.makedirs("jsonl_vi", exist_ok=True)
make_file(0, 10, "jsonl_vi/0_10.jsonl")
print("Created jsonl_vi/0_10.jsonl")

Created jsonl_vi/0_10.jsonl


In [8]:
# Upload + submit batch
with open("jsonl_vi/0_10.jsonl", "rb") as f:
    file_response = groq_client.files.create(file=f, purpose="batch")
file_id = file_response.id
print(f"Uploaded: {file_id}")

batch_response = groq_client.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=file_id,
)
print(f"Batch: {batch_response.id}, status: {batch_response.status}")

Uploaded: file_01knzvbgd0f5ma00282xv8n327
Batch: batch_01knzvbgpef5n814fhpgkvp23b, status: validating


In [10]:
# Check status (chay lai cho den khi completed)
result = groq_client.batches.retrieve(batch_response.id)
print(f"Status: {result.status}")
if result.status == "completed":
    print(f"Output file: {result.output_file_id}")

Status: completed
Output file: file_01knzvbmyeehgrn0w2n7nry0kq


In [11]:
# Fetch results + build summaries (dung build_summary de ghep data goc + LLM)
output = groq_client.files.content(result.output_file_id)
output.write_to_file("jsonl_vi/batch_results_test.jsonl")

with open("jsonl_vi/batch_results_test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        llm_text = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = build_summary(items[id], llm_text)

# Xem ket qua
for i in range(10):
    if items[i].summary:
        print(f"--- Item {i} ({items[i].category}, {items[i].price:,} VND) ---")
        print(items[i].summary)
        print()

--- Item 0 (Điện Tử - Công Nghệ, 2,376,000 VND) ---
Tiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Danh mục: Điện Tử - Công Nghệ
Thương hiệu: TEEMO PC
Mô tả: Pin Li‑ion tương thích hoàn chỉnh với Dell Vostro 14 5459, cung cấp thời gian sử dụng lên tới 6h tùy cấu hình.  
Thông số: Công suất chuẩn ±5%, thời gian sạc 8‑10h, bảo hành 6–12 tháng, bao gồm 100% thay mới khi lỗi trong thời gian bảo hành.

--- Item 1 (Thời Trang, 339,000 VND) ---
Tiêu đề: Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ
Danh mục: Thời Trang
Thương hiệu: LiLiLa
Mô tả: Áo len hoodie dày ấm, màu sắc tươi sáng, thiết kế trẻ trung, dễ phối đồ.  
Thông số: Chất len mềm mịn, tỉ mỉ trong từng đường may, fit freesize phù hợp nhiều vóc dáng.

--- Item 2 (Bách Hóa, 78,000 VND) ---
Tiêu đề: Trà lá xanh hương lá dứa Trần Quang (gói 500gr)
Danh mục: Bách Hóa
Thương hiệu: Trần Quang
Mô tả: Trà lá xanh hương lá dứa Trần Quang, 500gr, mang hương thơm đặc trưng của vù

## 4. Full Batch Processing (120K items)

Dung Batch class tu `pricer_vi/batch.py`. Batch class da tich hop `build_summary()` trong `apply_output()`.

**QUAN TRONG:** Reset summary cua test batch truoc khi chay full batch.

In [7]:
# Reset summaries from test batch
for item in items:
    item.summary = None

Batch.create(items)

Created 120 batches


In [8]:
Batch.run()

  0%|          | 0/120 [00:00<?, ?it/s]

Submitted 120 batches


In [21]:
# Chay cell nay nhieu lan cho den khi tat ca batches hoan thanh
Batch.fetch()

  0%|          | 0/120 [00:00<?, ?it/s]

ValueError: Expected a non-empty value for `file_id` but received None

In [23]:
from collections import Counter                                                                                                                        
                                                                                                                                                    
statuses = Counter()
no_output = []
for i, batch in enumerate(Batch.batches):                                                                                                              
    if batch.done:
        statuses["done"] += 1                                                                                                                          
        continue                                                                                                                                       
    try:
        response = groq_client.batches.retrieve(batch.batch_id)                                                                                        
        statuses[response.status] += 1                    
        if response.status == "completed" and not response.output_file_id:                                                                             
            no_output.append(i)
        if response.status in ("failed", "expired", "cancelled"):                                                                                      
            print(f"  Batch {i} ({batch.filename}): {response.status}")                                                                                
    except Exception as e:                                                                                                                             
        statuses["error"] += 1                                                                                                                         
        print(f"  Batch {i}: {e}")                                                                                                                     
                                                        
print(f"\nTong hop: {dict(statuses)}")                                                                                                                 
print(f"Batches khong co output_file_id: {len(no_output)}")


Tong hop: {'done': 75, 'completed': 45}
Batches khong co output_file_id: 44


In [25]:
import time

recovered = 0
still_missing = 0

for i, batch in enumerate(Batch.batches):
    if batch.done:
        continue

    try:
        response = groq_client.batches.retrieve(batch.batch_id)

        if response.output_file_id:
            batch.output_file_id = response.output_file_id
            batch.fetch_output()
            batch.apply_output()
            recovered += 1
        else:
            still_missing += 1
            if still_missing <= 3:
                print(f"  Batch {i} ({batch.filename}): completed but output_file_id=None")
                print(f"    error_file_id={response.error_file_id}")
                print(f"    request_counts={response.request_counts}")
    except Exception as e:
        print(f"  Batch {i}: {e}")

print(f"\nRecovered: {recovered}")
print(f"Still missing: {still_missing}")
print(f"Total done: {sum(1 for b in Batch.batches if b.done)}/120")

  Batch 75 (75000_76000.jsonl): completed but output_file_id=None
    error_file_id=file_01kp05bng4eckrk9mrjpsk48ae
    request_counts=RequestCounts(completed=0, failed=1000, total=1000)
  Batch 76 (76000_77000.jsonl): completed but output_file_id=None
    error_file_id=file_01kp05bqayefq8fbtcbzseyyq9
    request_counts=RequestCounts(completed=0, failed=1000, total=1000)
  Batch 77 (77000_78000.jsonl): completed but output_file_id=None
    error_file_id=file_01kp05btq2enhr0kc70jzh2dgr
    request_counts=RequestCounts(completed=0, failed=1000, total=1000)

Recovered: 1
Still missing: 44
Total done: 76/120


In [26]:
# Resubmit 44 failed batches                                                                                                                           
resubmitted = 0
                                                                                                                                                        
for i, batch in enumerate(Batch.batches):                                                                                                              
    if batch.done:
        continue                                                                                                                                       
                                                        
    # Re-upload file va submit lai                                                                                                                     
    try:
        batch.send_file()                                                                                                                              
        batch.submit_batch()                              
        resubmitted += 1
    except Exception as e:                                                                                                                             
        print(f"  Batch {i} ({batch.filename}): {e}")
                                                                                                                                                        
print(f"Resubmitted: {resubmitted} batches")                                                                                                           
Batch.save()
print("State saved")

Resubmitted: 44 batches
Saved 120 batches
State saved


In [27]:
import time                                                                                                                                            
                                                        
while True:
    Batch.fetch()
    finished = sum(1 for b in Batch.batches if b.done)                                                                                                 
    if finished == len(Batch.batches):
        print("All batches done!")                                                                                                                     
        break                                             
    print(f"Waiting 30s... ({finished}/{len(Batch.batches)})")
    time.sleep(30)  

  0%|          | 0/120 [00:00<?, ?it/s]

Finished 76 of 120 batches
Waiting 30s... (76/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 77 of 120 batches
Waiting 30s... (77/120)


  0%|          | 0/120 [00:00<?, ?it/s]

...

Finished 119 of 120 batches
Waiting 30s... (119/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 119 of 120 batches
Waiting 30s... (119/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 120 of 120 batches
All batches done!


In [28]:
# Save state (de resume neu kernel bi ngat)
Batch.save()

Saved 120 batches


In [29]:
# Fix 3 partial batches                                                                                                                                
import os
                                                                                                                                                        
partial_indices = [73, 74, 119]                                                                                                                        

for idx in partial_indices:                                                                                                                            
    batch = Batch.batches[idx]                            
    print(f"Fixing batch {idx} ({batch.filename}): done={batch.done}")
                                                                                                                                                        
    # Xoa output cu
    output_file = batch.output_dir / batch.filename                                                                                                    
    if output_file.exists():                              
        os.remove(output_file)
        print(f"  Deleted {output_file}")                                                                                                              

    # Reset state                                                                                                                                      
    batch.done = False                                    
    batch.output_file_id = None
                                                                                                                                                        
    # Resubmit
    batch.send_file()                                                                                                                                  
    batch.submit_batch()                                  
    print(f"  Resubmitted: batch_id={batch.batch_id}")
                                                                                                                                                        
Batch.save()
print(f"\nResubmitted 3 batches. State saved.")

Fixing batch 73 (73000_74000.jsonl): done=True
  Deleted output_vi/73000_74000.jsonl
  Resubmitted: batch_id=batch_01kp0beazpff0vsb4yjs9y2ed9
Fixing batch 74 (74000_75000.jsonl): done=True
  Deleted output_vi/74000_75000.jsonl
  Resubmitted: batch_id=batch_01kp0becx5ffx98jc5a7tbwdqy
Fixing batch 119 (119000_120000.jsonl): done=True
  Deleted output_vi/119000_120000.jsonl
  Resubmitted: batch_id=batch_01kp0beefqfgpvb88hwfrt36h1
Saved 120 batches

Resubmitted 3 batches. State saved.


In [30]:
import time                                                                                                                                            
                                                        
while True:
    Batch.fetch()
    finished = sum(1 for b in Batch.batches if b.done)                                                                                                 
    if finished == len(Batch.batches):
        print("All batches done!")                                                                                                                     
        break                                             
    print(f"Waiting 30s... ({finished}/{len(Batch.batches)})")
    time.sleep(30)  

  0%|          | 0/120 [00:00<?, ?it/s]

Finished 117 of 120 batches
Waiting 30s... (117/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 117 of 120 batches
Waiting 30s... (117/120)


  0%|          | 0/120 [00:00<?, ?it/s]

...

Finished 119 of 120 batches
Waiting 30s... (119/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 119 of 120 batches
Waiting 30s... (119/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 120 of 120 batches
All batches done!


In [ ]:
# Resume tu session truoc (neu can):
# 1. Chay lai tu cell 1 den cell "Assign IDs"
# 2. Uncomment va chay cell nay
# 3. Tiep tuc Batch.fetch()

# Batch.load(items)

## 5. Kiem tra ket qua

In [31]:
# Kiem tra missing summaries
missing = [i for i, item in enumerate(items) if not item.summary]
print(f"Missing summaries: {len(missing)}")
if missing:
    print(f"First 10 missing IDs: {missing[:10]}")

Missing summaries: 0


In [32]:
# Xem vi du summary tu nhieu categories
for i in [0, 100, 1000, 5000, 50000, 100000]:
    if i < len(items) and items[i].summary:
        print(f"--- Item {i} ({items[i].category}, {items[i].price:,} VND) ---")
        print(items[i].summary)
        print()

--- Item 0 (Điện Tử - Công Nghệ, 2,376,000 VND) ---
Tiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Danh mục: Điện Tử - Công Nghệ
Thương hiệu: TEEMO PC
Mô tả: Pin Li‑ion mới, hoàn toàn tương thích với Laptop Dell Vostro 14 5459, cung cấp công suất tiêu chuẩn ±5% và thời gian sử dụng từ 2h-6h tùy cell.  
Thông số: Độ bền lâu dài, thời gian bảo hành 6-12 tháng, có thẻ bảo hành chính hãng và hỗ trợ đổi mới 100% trong thời gian bảo hành.

--- Item 100 (Mẹ và Bé, 59,000 VND) ---
Tiêu đề: Patch PVC Velcro Zombie dán ba lô túi xách
Danh mục: Mẹ và Bé
Thương hiệu: OEM
Mô tả: Patch PVC velcro zombie thiết kế chiến thuật, dễ dán và gắn vào quần áo, túi xách, balo, nón.  
Thông số: Chất liệu dẻo, sắc nét không biến dạng, mặt sau có gai giúp dán chắc chắn và chịu được mọi hoạt động.

--- Item 1000 (Điện Tử - Công Nghệ, 222,000 VND) ---
Tiêu đề: Ugreen UG11672DV101TK 1M màu Đen Cáp tín hiệu DVI 24 + 1 - HÀNG CHÍNH HÃNG
Danh mục: Điện Tử - Công Ngh

## 6. Build prompts + Clean up + Push to HF Hub

In [33]:
# Build prompt from summary
for item in items:
    if item.summary:
        item.make_prompt(item.summary)

# Verify prompt format
print("=== Example prompt ===")
print(items[0].prompt)

=== Example prompt ===
Sản phẩm này giá bao nhiêu?

Tiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Danh mục: Điện Tử - Công Nghệ
Thương hiệu: TEEMO PC
Mô tả: Pin Li‑ion mới, hoàn toàn tương thích với Laptop Dell Vostro 14 5459, cung cấp công suất tiêu chuẩn ±5% và thời gian sử dụng từ 2h-6h tùy cell.  
Thông số: Độ bền lâu dài, thời gian bảo hành 6-12 tháng, có thẻ bảo hành chính hãng và hỗ trợ đổi mới 100% trong thời gian bảo hành.

Giá: 2376000


In [34]:
# Clean up: remove fields not needed in final dataset
for item in items:
    item.full = None
    item.brand = None
    item.id = None

# Verify
print(items[0].model_dump())

{'title': 'Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870', 'category': 'Điện Tử - Công Nghệ', 'price': 2376000, 'full': None, 'brand': None, 'summary': 'Tiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870\nDanh mục: Điện Tử - Công Nghệ\nThương hiệu: TEEMO PC\nMô tả: Pin Li‑ion mới, hoàn toàn tương thích với Laptop Dell Vostro 14 5459, cung cấp công suất tiêu chuẩn ±5% và thời gian sử dụng từ 2h-6h tùy cell.  \nThông số: Độ bền lâu dài, thời gian bảo hành 6-12 tháng, có thẻ bảo hành chính hãng và hỗ trợ đổi mới 100% trong thời gian bảo hành.', 'prompt': 'Sản phẩm này giá bao nhiêu?\n\nTiêu đề: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870\nDanh mục: Điện Tử - Công Nghệ\nThương hiệu: TEEMO PC\nMô tả: Pin Li‑ion mới, hoàn toàn tương thích với Laptop Dell Vostro 14 5459, cung cấp công suất tiêu chuẩn ±5% và thời gian sử dụng từ 2h-6h tùy cell.  \nThông s

In [35]:
# Push to HuggingFace Hub
username = "SeanSunny"
output_dataset = f"{username}/items_tv_v4"

# Split: giu nguyen thu tu tu items_raw_tv_v4 (110K train / 5K val / 5K test)
train = items[:110_000]
val = items[110_000:115_000]
test = items[115_000:]

print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")
Item.push_to_hub(output_dataset, train, val, test)
print(f"Pushed to {output_dataset}")

Train: 110,000, Val: 5,000, Test: 5,000


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/110 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to SeanSunny/items_tv_v4


## Done!

Dataset `SeanSunny/items_tv_v4` da co summary (5 truong: Tieu de/Danh muc/Thuong hieu tu data goc + Mo ta/Thong so tu LLM) va prompt (cho fine-tuning).

San sang cho Day 3 (Baseline ML) va Day 4 (DNN + Frontier LLM).